# Transverse-Field Ising Model Cross-Method Benchmark

This notebook exercises the non-molecule expert-mode API with a one-dimensional transverse-field Ising model (TFIM):

`H = -J sum_i Z_i Z_{i+1} - g sum_i X_i`

It compares exact diagonalization, VQE, VarQITE, and QPE across a small transverse-field sweep. The goal is to validate the generic qubit-Hamiltonian path rather than the chemistry pipeline.

In [ ]:
from __future__ import annotations

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pennylane as qml
from IPython.display import display

from qite import run_qite
from qpe import run_qpe
from vqe import run_vqe

N_QUBITS = 4
J = 1.0
FIELD_GRID = [0.25, 0.75, 1.0, 1.5, 2.0]
REFERENCE_STATE = [0] * N_QUBITS

# VQE parameters
VQE_STEPS = 75

# QITE parameters
QITE_STEPS = 50
QITE_DTAU = 0.05
QITE_REG = 1e-4

# QPE parameters
QPE_ANCILLAS = 5
QPE_TIME = 1.0
QPE_TROTTER_STEPS = 2

# Ansatz parameters
ANSATZ_NAME = "auto"
ANSATZ_KWARGS = {"layers": 4}


In [ ]:
fig, ax = plt.subplots(figsize=(8, 2.6))

site_x = np.arange(N_QUBITS, dtype=float)
for wire, x in enumerate(site_x):
    ax.add_patch(plt.Rectangle((x - 0.18, -0.18), 0.36, 0.36, facecolor="#f7f7f7", edgecolor="#222222", linewidth=1.5))
    ax.text(x, 0.0, f"q{wire}", ha="center", va="center", fontsize=10)
    ax.arrow(x, 0.42, 0.0, -0.18, width=0.015, head_width=0.08, head_length=0.06, color="#b56576", length_includes_head=True)
    ax.text(x, 0.62, "X field", ha="center", va="bottom", fontsize=8, color="#b56576")

for wire in range(N_QUBITS - 1):
    ax.plot([wire + 0.2, wire + 0.8], [0, 0], color="#355070", linewidth=2.5)
    ax.text(wire + 0.5, -0.32, "ZZ", ha="center", va="top", fontsize=9, color="#355070")

ax.set_title(rf"TFIM chain: $-J\sum Z_iZ_{{i+1}} - g\sum X_i$;  (N={N_QUBITS})")
ax.set_xlim(-0.5, N_QUBITS - 0.5)
ax.set_ylim(-0.65, 0.95)
ax.axis("off")
fig.tight_layout()
plt.show()

In [ ]:
def transverse_field_ising_hamiltonian(num_qubits: int, *, j: float, g: float) -> qml.Hamiltonian:
    coeffs = []
    ops = []
    for wire in range(num_qubits - 1):
        coeffs.append(-float(j))
        ops.append(qml.PauliZ(wire) @ qml.PauliZ(wire + 1))
    for wire in range(num_qubits):
        coeffs.append(-float(g))
        ops.append(qml.PauliX(wire))
    return qml.Hamiltonian(coeffs, ops)


def exact_ground_energy(hamiltonian: qml.Hamiltonian, num_qubits: int) -> float:
    matrix = qml.matrix(hamiltonian, wire_order=list(range(num_qubits)))
    return float(np.linalg.eigvalsh(np.asarray(matrix, dtype=complex))[0].real)


In [ ]:
rows = []

for g in FIELD_GRID:
    H = transverse_field_ising_hamiltonian(N_QUBITS, j=J, g=g)
    exact = exact_ground_energy(H, N_QUBITS)

    vqe_res = run_vqe(
        molecule=f"TFIM_N{N_QUBITS}_g{g:g}",
        hamiltonian=H,
        num_qubits=N_QUBITS,
        reference_state=REFERENCE_STATE,
        ansatz_name=ANSATZ_NAME,
        ansatz_kwargs=ANSATZ_KWARGS,
        optimizer_name="Adam",
        steps=VQE_STEPS,
        plot=False,
        force=False,
    )

    qite_res = run_qite(
        molecule=f"TFIM_N{N_QUBITS}_g{g:g}",
        hamiltonian=H,
        num_qubits=N_QUBITS,
        reference_state=REFERENCE_STATE,
        ansatz_name=ANSATZ_NAME,
        ansatz_kwargs=ANSATZ_KWARGS,
        steps=QITE_STEPS,
        dtau=QITE_DTAU,
        reg=QITE_REG,
        plot=False,
        show=False,
        force=False,
    )

    qpe_res = run_qpe(
        molecule=f"TFIM_N{N_QUBITS}_g{g:g}",
        hamiltonian=H,
        hf_state=np.array(REFERENCE_STATE, dtype=int),
        system_qubits=N_QUBITS,
        n_ancilla=QPE_ANCILLAS,
        t=QPE_TIME,
        trotter_steps=QPE_TROTTER_STEPS,
        shots=None,
        plot=False,
        force=False,
    )

    rows.append(
        {
            "g": float(g),
            "exact_ground_energy": exact,
            "vqe_energy": float(vqe_res["energy"]),
            "qite_energy": float(qite_res["energy"]),
            "qpe_energy": float(qpe_res["energy"]),
            "vqe_abs_error": abs(float(vqe_res["energy"]) - exact),
            "qite_abs_error": abs(float(qite_res["energy"]) - exact),
            "qpe_abs_error": abs(float(qpe_res["energy"]) - exact),
            "qpe_best_bitstring": qpe_res["best_bitstring"],
            "qpe_best_probability": float(qpe_res["probs"][qpe_res["best_bitstring"]]),
            "vqe_runtime_s": float(vqe_res.get("compute_runtime_s", np.nan)),
            "qite_runtime_s": float(qite_res.get("compute_runtime_s", np.nan)),
            "qpe_runtime_s": float(qpe_res.get("compute_runtime_s", np.nan)),
        }
    )

summary_df = pd.DataFrame(rows)
display(summary_df.round(8))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))

series = {
    "exact": {"energy": "exact_ground_energy", "color": "#222222"},
    "VQE": {"energy": "vqe_energy", "error": "vqe_abs_error", "color": "#355070"},
    "VarQITE": {"energy": "qite_energy", "error": "qite_abs_error", "color": "#b56576"},
    "QPE dominant phase": {"energy": "qpe_energy", "error": "qpe_abs_error", "color": "#6d597a"},
}

handles = []
for label, spec in series.items():
    (line,) = axes[0].plot(
        summary_df["g"],
        summary_df[spec["energy"]],
        marker="o",
        color=spec["color"],
        label=label,
    )
    handles.append(line)
axes[0].set_title("Energy vs transverse field")
axes[0].set_xlabel("g")
axes[0].set_ylabel("energy")
axes[0].grid(True, alpha=0.3)

for label, spec in series.items():
    if "error" not in spec:
        continue
    axes[1].semilogy(
        summary_df["g"],
        summary_df[spec["error"]],
        marker="o",
        color=spec["color"],
        label=label,
    )
axes[1].set_title("Absolute error to exact ground")
axes[1].set_xlabel("g")
axes[1].set_ylabel("absolute error")
axes[1].grid(True, alpha=0.3)

fig.suptitle(f"TFIM expert-mode benchmark (N={N_QUBITS}, J={J})", y=0.99)
fig.legend(handles=handles, loc="upper center", ncol=len(handles), bbox_to_anchor=(0.5, 0.92))
fig.tight_layout(rect=[0, 0, 1, 0.82])
plt.show()


## Notes

- VQE and VarQITE are ground-state methods here, so their energies should be read against the exact ground state.
- QPE is initialized from a computational basis state, not an exact eigenstate. Its dominant phase is a useful spectral diagnostic, but it is not guaranteed to select the ground state unless the initial state has strong ground-state overlap.
- Increase `N_QUBITS`, ansatz expressivity, or the field grid only after the four-qubit notebook behavior is stable.
- TFIM uses `ANSATZ_NAME = "auto"`; the selector should resolve this Hamiltonian to `TFIM-HVA`, whose layers alternate the same ZZ and X operator families that appear in the Hamiltonian. The selected ansatz is reported in the run result.
- The VarQITE defaults use a smaller imaginary-time step and explicit regularization because larger-step settings can stall near and above the critical region.
